# Import Libraries

In [140]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
# Weather data library
from meteostat import Stations, Daily
from datetime import datetime
from pyathena import connect
import warnings
warnings.filterwarnings("ignore")

# Connect SageMaker to S3

In [141]:
#Create S3 client using boto3
#This allows the notebook to communicate with S3
s3 = boto3.client('s3')

#Define your S3 bucket name where your datasets are stored
bucket = "vegetation-risk-ml"

#list files inside bucket
response = s3.list_objects_v2(Bucket=bucket)

#Print the names of the files in the bucket
for obj in response['Contents']:
    print(obj['Key'])

athena-results/
athena-results/1155328f-8cb0-44fe-9051-ad6d337c42e1.csv
athena-results/1155328f-8cb0-44fe-9051-ad6d337c42e1.csv.metadata
athena-results/19f60d9f-3a56-4e8d-b77b-1690beab649b.txt
athena-results/206599a9-6d5a-4a5e-84a8-d876256dd33a.txt
athena-results/3699d66f-4438-44f8-b077-e431282eb850.csv
athena-results/3699d66f-4438-44f8-b077-e431282eb850.csv.metadata
athena-results/36b7736e-ab3a-4dc8-9a9f-7815c198c365.txt
athena-results/3824b27e-f0b3-4696-bcfa-414dbdc4bd7c.csv
athena-results/3824b27e-f0b3-4696-bcfa-414dbdc4bd7c.csv.metadata
athena-results/3c4810db-80d6-4f4d-a0e4-33f522e69697.txt
athena-results/4294d17c-63ef-40b6-95aa-98dd3b8757f1.csv
athena-results/4294d17c-63ef-40b6-95aa-98dd3b8757f1.csv.metadata
athena-results/50ff53ab-efe7-4017-b440-e2d6a7a4bcbf.txt
athena-results/531dcfd9-2b9a-4c2d-a0f4-4b0aa0493326.csv
athena-results/531dcfd9-2b9a-4c2d-a0f4-4b0aa0493326.csv.metadata
athena-results/5627d404-f98a-4109-9280-e269a1a6c59c.txt
athena-results/5638a933-f724-479f-9f06-f877

# **DATA PREPARATION**

**Define s3 path**

In [142]:
#Define S3 fi
forest_path  = "s3://vegetation-risk-ml/processed/forest/combined_forest_data.csv"
fire_path    = "s3://vegetation-risk-ml/raw/fire/California_Historic_Fire_data.csv"
weather_path = "s3://vegetation-risk-ml/raw/weather/california_weather_data.csv"

**Load datasets**

In [143]:
combined_forest = pd.read_csv(forest_path, low_memory=False)
print("Forest dataset shape :", combined_forest.shape)
fire = pd.read_csv(fire_path, low_memory=False)
print("Fire dataset shape :", fire.shape)
weather = pd.read_csv(weather_path, low_memory=False)
print("Weather dataset shape :", weather.shape)

print("Datasets loaded successfully")

Forest dataset shape : (1687167, 11)
Fire dataset shape : (3372, 23)
Weather dataset shape : (159235, 12)
Datasets loaded successfully


# DATA CLEANING AND PREPROCESSING

### Forest dataset cleaning

We will handle missing values and drop the unnecessary columns so that the dataset represents only relevent columns which directly affects trimming prioritization.This stage will prepare our dataset for integration.

Here only variables that influence vegetation density and wildfire potential are retained.

**Drop unnecessary columns**

 We drop plt_cn,spsd to avoid noise in the data and reduce memory usage.From exploration,we know that REGIONAL_DRYBIOT and REGIONAL_DRYBIOM are highly correlated so we keep only one of them.

In [ ]:
#select most relevent column 
combined_forest=combined_forest[["DIA", "HT", "SLOPE", "REGIONAL_DRYBIOT","LAT", "LON","COUNTYCD"]]  #tree size,height,terrain,direction,biomass

#check the shape of the dataset
combined_forest.shape

(1687167, 4)

**Remove Duplicates**

In [145]:
#remove duplicates
combined_forest = combined_forest.drop_duplicates()
# Verify
print("Duplicates remaining:", combined_forest.duplicated().sum())

Duplicates remaining: 0


**Handle missing valus**

To handle missing values, median is used because biomass is skewed and avoids distortion from outliers.

In [ ]:
#fill missing values for biomass
combined_forest = combined_forest.dropna(subset=["LAT", "LON"])
combined_forest["REGIONAL_DRYBIOT"] = combined_forest["REGIONAL_DRYBIOT"].fillna(combined_forest["REGIONAL_DRYBIOT"].median())
combined_forest["SLOPE"] = combined_forest["SLOPE"].fillna(combined_forest["SLOPE"].median())
combined_forest["DIA"] = combined_forest["DIA"].fillna(combined_forest["DIA"].median()) 
combined_forest["HT"] = combined_forest["HT"].fillna(combined_forest["HT"].median())                                                     

# Verify
print("Missing values:\n", combined_forest.isnull().sum())

Missing values:
 DIA                 0
HT                  0
SLOPE               0
REGIONAL_DRYBIOT    0
dtype: int64


**Handle outlier**

In [147]:
#handle unrealistic value of tree diameter and height
combined_forest = combined_forest[(combined_forest["DIA"] > 0) & (combined_forest["HT"] > 0)]

In [148]:
#Verify the resulting dataset
combined_forest.head()

,DIA,HT,SLOPE,REGIONAL_DRYBIOT
0,3.9,43.0,50.0,495.93962
1,3.9,43.0,60.0,495.93962
2,3.9,43.0,40.0,495.93962
3,24.1,95.0,50.0,495.93962
4,24.1,95.0,60.0,495.93962


### Fire dataset cleaning

Here we will keep valid geographic coordinates and meaningful fire size measurements only.

**Select Relevent Columns**

In [149]:
#select relevant features only
fire=fire[["incident_acres_burned", "incident_longitude", "incident_latitude", "incident_date_created","incident_county"]]

In [ ]:
#remove fires with invalid coordinates (outside California)
fire = fire[(fire["incident_latitude"].between(32, 42)) &(fire["incident_longitude"].between(-124, -114))]

#Remove fires with missing coordinates
fire = fire.dropna(subset=["incident_latitude", "incident_longitude"])

#Remove fires with zero size
fire = fire[fire["incident_acres_burned"] > 0]

#Convert datecolumn to datetime and remove timezone
fire["incident_date_created"] = pd.to_datetime(fire["incident_date_created"], errors="coerce").dt.date

# Remove missing county
fire = fire.dropna(subset=["incident_county", "incident_date_created"])

# Reset index
fire = fire.reset_index(drop=True)

#display results
print("Cleaned fire dataset shape:", fire.shape)


Cleaned fire dataset shape: (3294, 5)


In [151]:
#check for any remaining missing value
print(fire.isnull().sum())
fire.head()

incident_acres_burned    0
incident_longitude       0
incident_latitude        0
incident_date_created    0
incident_county          9
dtype: int64


,incident_acres_burned,incident_longitude,incident_latitude,incident_date_created,incident_county
0,37.0,-122.309000,40.774000,2017-10-31 11:22:00,Shasta
1,122.0,1.000000,1.000000,2009-05-24 14:56:00,San Diego
2,407.0,-118.016510,36.602575,2013-02-24 08:16:00,Inyo
3,30.0,-116.941311,34.288877,2013-04-20 17:30:00,San Bernardino
4,274.0,-119.635004,37.116295,2013-04-30 12:59:00,Madera


#### Weather dataset cleaning

We will select temperature ,precipitation and wind speed that influence vegetation growth and fire risk. Here missing values are imputed to ensure consistent weather observations.

In [152]:
#Select most relevent columns
weather = weather[["tmin","tmax","time", "tavg", "prcp", "wspd", "station"]]

In [ ]:
#Remane column  for consistency
weather.rename(columns={"latitude": "LAT","longitude": "LON"}, inplace=True)

# Drop missing location
weather = weather.dropna(subset=["LAT", "LON"])

# Convert time column
weather["time"] = pd.to_datetime(weather["time"]).dt.date

# Fill missing rainfall
weather["prcp"] = weather["prcp"].fillna(0)

# Fill wind speed
weather["wspd"] = weather["wspd"].fillna(weather["wspd"].median())

# Remove unrealistic temperatures
weather = weather[(weather["tavg"] > -40) & (weather["tavg"] < 60)]

# Fill missing average temperature
weather["tavg"] = weather["tavg"].fillna((weather["tmin"] + weather["tmax"])/2)

# We can drop tmax and tmin as we have tavg, and they are highly correlated
weather = weather[["time", "tavg", "prcp", "wspd", "station"]]

In [154]:
#Verify the resulting dataset
print("Cleaned weather dataset shape:", weather.shape)
#check for any remaining missing values
print("Missing values in each column:")
print(weather.isnull().sum())
print("First few rows of the cleaned dataset:")
weather.head()

Cleaned weather dataset shape: (152333, 5)
Missing values in each column:
time       0
tavg       0
prcp       0
wspd       0
station    0
dtype: int64
First few rows of the cleaned dataset:


,time,tavg,prcp,wspd,station
0,2020-11-05,13.5,0.0,10.3,69007
1,2020-11-06,12.1,0.0,10.3,69007
2,2020-11-07,9.9,0.0,10.3,69007
3,2020-11-08,9.1,0.0,10.3,69007
4,2020-11-09,5.5,0.0,10.3,69007


#### **Save the cleaned dataset to S3**

The cleaned datasets are then store back in S3 in separete folder processed/ for future analysis.

In [155]:
combined_forest.to_csv("s3://vegetation-risk-ml/processed/forest/forest_cleaned.csv",index=False)

fire.to_csv("s3://vegetation-risk-ml/processed/fire/fire_cleaned.csv",index=False)

weather.to_csv("s3://vegetation-risk-ml/processed/weather/weather_cleaned.csv",index=False)

print("Cleaned datasets saved successfully in S3")

Cleaned datasets saved successfully in S3


Next,to merge all three cleaned datasets, we are using Athena.

Athena is used to perform  data merging of all three cleaned datasets and aggregation directly on S3 without loading large datasets into memory.This resolves the high memory usage and slow processing.The final unified feature table will be used for model training and dashboard visualization.

# Connect SageMaker to Athena

In [156]:
#Create folfer in S3 to save athena results
s3 = boto3.client("s3")

s3.put_object(Bucket="vegetation-risk-ml",Key="athena-results/")
print("athena-results folder created in S3")

athena-results folder created in S3


In [157]:
#verify the athena results folder is created
response = s3.list_objects_v2(Bucket="vegetation-risk-ml",Prefix="athena-results/")
print("Folder exists:", "athena-results/" in [obj["Key"] for obj in response.get("Contents", [])])

Folder exists: True


In [158]:

#Query results are stored in s3 in this directory
s3_staging_dir = "s3://vegetation-risk-ml/athena-results/"

# Athena connection
conn = connect(s3_staging_dir=s3_staging_dir,region_name="us-east-1")

print("Connected to Athena")

Connected to Athena


# Create Table for Forest Dataset

In [ ]:
#drop old tables if exist
drop_query = """
DROP TABLE IF EXISTS vegetation_ml.forest_cleaned
"""
pd.read_sql(drop_query, conn)
print("Old table dropped if any")

Old table dropped


In [160]:
#create forest table
forest_query = """
CREATE EXTERNAL TABLE IF NOT EXISTS vegetation_ml.forest_cleaned (
    DIA DOUBLE,
    HT DOUBLE,
    SLOPE DOUBLE,
    REGIONAL_DRYBIOT DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
LOCATION 's3://vegetation-risk-ml/processed/forest/'
TBLPROPERTIES ("skip.header.line.count"="1","use.null.for.invalid.data"="true","use.null.for.invalid.data"="true");
"""

pd.read_sql(forest_query, conn)
print("Forest table created successfully")

Forest table created successfully


# Create Table for Weather Dataset

In [161]:
#drop if table already exists
drop_query = """
DROP TABLE IF EXISTS vegetation_ml.weather_cleaned
"""
pd.read_sql(drop_query, conn)
print("Old table dropped")

Old table dropped


In [162]:
weather_query = """
CREATE EXTERNAL TABLE IF NOT EXISTS vegetation_ml.weather_cleaned (
    time STRING,
    tavg DOUBLE,
    prcp DOUBLE,
    wspd DOUBLE,
    station STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
LOCATION 's3://vegetation-risk-ml/processed/weather/'
TBLPROPERTIES ("skip.header.line.count"="1","use.null.for.invalid.data"="true");
"""
pd.read_sql(weather_query, conn)
print("Weather table created successfully")


Weather table created successfully


# Create Table for Fire Dataset

In [163]:
#drop if table exists
drop_query = """
DROP TABLE IF EXISTS vegetation_ml.fire_cleaned
"""
pd.read_sql(drop_query, conn)

""


In [ ]:
fire_query = """
CREATE EXTERNAL TABLE IF NOT EXISTS vegetation_ml.fire_cleaned (
    incident_acres_burned DOUBLE,
    incident_longitude DOUBLE,
    incident_latitude DOUBLE,
    incident_date_created DATE,
    incident_county STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
LOCATION 's3://vegetation-risk-ml/processed/fire/'
TBLPROPERTIES ("skip.header.line.count"="1","use.null.for.invalid.data"="true");
"""
pd.read_sql(fire_query, conn)
print("Fire table created successfully")

Fire table created successfully


**Verify tables**

In [165]:

# Check if tables exist and data is accessible
for table in ["forest_cleaned", "weather_cleaned", "fire_cleaned"]:
    query = f"SELECT * FROM vegetation_ml.{table} LIMIT 5"
    df = pd.read_sql(query, conn)
    print(f"Sample data from {table}:")
    print(df.head())
    print("\n")

Sample data from forest_cleaned:
        dia       ht  slope  regional_drybiot
0  675267.0  11839.0    3.9              43.0
1  675267.0  11839.0    3.9              43.0
2  675267.0  11839.0    3.9              43.0
3  675268.0  11839.0   24.1              95.0
4  675268.0  11839.0   24.1              95.0


Sample data from weather_cleaned:
         time  tavg  prcp  wspd station
0  2020-11-05  13.5   0.0  10.3   69007
1  2020-11-06  12.1   0.0  10.3   69007
2  2020-11-07   9.9   0.0  10.3   69007
3  2020-11-08   9.1   0.0  10.3   69007
4  2020-11-09   5.5   0.0  10.3   69007


Sample data from fire_cleaned:
   incident_acres_burned  incident_longitude  incident_latitude  \
0                   37.0         -122.309000          40.774000   
1                  122.0            1.000000           1.000000   
2                  407.0         -118.016510          36.602575   
3                   30.0         -116.941311          34.288877   
4                  274.0         -119.635004   

# Create Unified Feature table in Athena

Now we create the final Feature table by merging all three tables. This final table becomes the single source of truth for modelling and dashboard analysis.

In [ ]:

CREATE TABLE vegetation_ml.vegetation_risk_dataset WITH (format='PARQUET',external_location='s3://vegetation-risk-ml/final/') AS
SELECT
    f.DIA,              --Tree diameter
    f.HT,               --Tree height
    f.SLOPE,            --Terrain slope
    f.REGIONAL_DRYBIOT, --Vegetation biomass

    w.avg_temp,    --Average temperature
    w.avg_rain,    --Average precipitation
    w.avg_wind,

    fi.fire_count,     --Number of fires in the area
    fi.max_fire_size   --Maximum size of fires in the area
    fi.sample_lat,   --Latitude of fire incidents
    fi.sample_lon,   --Longitude of fire incidents
    fi.fire_year          --Year of fire incidents

FROM vegetation_ml.forest_cleaned f

LEFT JOIN    --Aggregate weather data to get average conditions
(
    SELECT
        AVG(tavg) AS avg_temp,
        AVG(prcp) AS avg_rain,
        AVG(wspd) AS avg_wind
    FROM vegetation_ml.weather_cleaned
) w
ON TRUE

LEFT JOIN     --Aggregate time-based fire data to get fire frequency, size and location
(
    SELECT
        incident_county,
        COUNT(*) AS fire_count,
        AVG(incident_acres_burned) AS incident_acres_burned,
        AVG(incident_latitude) AS incident_latitude,
        AVG(incident_longitude) AS incident_longitude,
        MIN(incident_date_created) AS incident_date_created
    FROM vegetation_ml.fire_cleaned
    GROUP BY incident_county
) fi
ON TRUE;

In [ ]:
#verify if table is created and data is populated
query = "SELECT * FROM vegetation_ml.vegetation_risk_dataset LIMIT 5"
vegetation_risk_dataset = pd.read_sql(query, conn)


#Verify the resulting dataset
print(vegetation_risk_dataset.shape)
vegetation_risk_dataset.head()

**Load Final Feature Table from S3 into SageMaker**

The unified feature table (source of truth) created by Athena is loaded from the S3 final/ folder into SageMaker for feature engineering and model training.

In [ ]:
vegetation_risk_data = pd.read_parquet("s3://vegetation-risk-ml/final/")

# **Feature Engineering**

# Feature Creation

Here we will create new features and transform the existing features for our machine learning model.

**Fuel Mositure Risk**

This featutee captures vegetation dryness(hot weather and no rain).The higher value means drier the vegetation condition,which increase wildfire risk if ignition happens. Its calculated as fuel moisture risk=temperatue/(rainfall+1).

Area with high temperature and low rainfall are where dry vegetation accumulates.Such plots need to be trimmed first as a small ignition can trigger large fire.

In [ ]:
# Calculate fuel moisture risk as the ratio of average temperature to average rainfall,#=1 to avoid division by zero
vegetation_risk_data["fuel_moisture_risk"] = vegetation_risk_data["avg_temp"] / (vegetation_risk_data["avg_rain"] + 1)
vegetation_risk_data.value_counts("fuel_moisture_risk").head()

**Fire Recurrence Risk**

Locations with repeated fire history may require more aggressive vegetation management, as those areas are more likely to burn again. We calculate this by Fire recurrence= number of fires in near location.

In [ ]:
# Log transform fire recurrence to compress the right-skewed distribution
vegetation_risk_data["fire_recurrence"] = np.log1p(vegetation_risk_data["fire_count"])

**Vegetation Fuel Load Index**

A dense stand of large, tall trees with high biomass contains far more burnable material than sparse, small shrubs. This index quantifies the total combustible fuel load at each plot by combining tree physical size with biomass.

This measures howmuch burnable vegetation exist. This is measured as Fuel load=tree size*toal biomass

In [ ]:
vegetation_risk_data["fuel_load_index"] =np.log1p(vegetation_risk_data["DIA"] * vegetation_risk_data["HT"] *(vegetation_risk_data["REGIONAL_DRYBIOT"]))

Combines tree size and biomass.Higher value indicates dense vegetation that could act as a wildfirefuel.

**Combined Wildfire Risk Score**

Here we create risk score by combining vegetation fuel load, moisture risk, and fire recurrence.Since fuel load index is high in value and can dominate,we will normalise using min max scaler for finding the fire risk score.

In [ ]:
#normalize before combining into wildfire_risk_score
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
cols_to_scale = ["fuel_load_index", "fuel_moisture_risk", "fire_recurrence"]
vegetation_risk_data[cols_to_scale] = scaler.fit_transform(vegetation_risk_data[cols_to_scale])

This score becomes the basis for creating the target variable (trim_priority).

In [ ]:
vegetation_risk_data["wildfire_risk_score"] = (0.4 * vegetation_risk_data["fuel_load_index"] +0.3 * vegetation_risk_data["fuel_moisture_risk"] +0.3 * vegetation_risk_data["fire_recurrence"])

**Create Target Variable**

The final target variable classifies vegetation trimming priority into three categories.The wildfire risk score score feeds directly into pd.qcut to create the Low / Medium / High bins.The classification helps to understand the urgency for trimming.

In [ ]:
vegetation_risk_data["trim_priority"] = pd.qcut(vegetation_risk_data["wildfire_risk_score"],q=3,labels=["Low","Medium","High"])

vegetation_risk_data["trim_priority"].value_counts(normalize=True)

# Feature Extraction

In [ ]:
# Extract year and month from merged dataset
vegetation_risk_data["fire_year"] = pd.to_datetime(vegetation_risk_data["incident_date_created"]).dt.year

In [ ]:
vegetation_risk_data["fire_month"] = pd.to_datetime(vegetation_risk_data["incident_date_created"]).dt.month

# Feature Transformation (log transformation)

To reduce skweness of data we apply log transormation to fire size,biomass 

In [ ]:
# Log transform fire size (use correct column name)
vegetation_risk_data["log_fire_size"] = np.log1p(vegetation_risk_data["incident_acres_burned"])

# Save Final Modeling dataset in S3

In [ ]:
vegetation_risk_data.to_csv("s3://vegetation-risk-ml/final/vegetation_ml_dataset.csv",index=False)

print("Final ML dataset saved")
print(f"Shape: {vegetation_risk_data.shape}")
print(f"Columns: {list(vegetation_risk_data.columns)}")